# Demo — CloudTrail Audit Evidence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/demos/demo-cloudtrail-audit/demo-cloudtrail-audit.ipynb)

This notebook is a follow-along demo.

Day 1 — Block 3: Securing the Agent (Identity and Least Privilege)

Shows how three-identity separation creates attributable audit logs.
Every agent action maps back to the specific workload identity and caller.

In [ ]:
# Install required dependencies
!pip install boto3 --quiet

In [ ]:
import json

def print_section(title: str):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def show_cloudtrail_event():
    """A simulated CloudTrail event proving identity attribution."""
    print_section("CloudTrail Event — Agent Tool Invocation")

    event = {
        "eventVersion": "1.08",
        "userIdentity": {
            "type": "AssumedRole",
            "principalId": "AROA1234567890EXAMPLE:session-99x-isolated",
            "arn": "arn:aws:sts::111122223333:assumed-role/ProdAgentRole/session-99x-isolated",
        },
        "eventTime": "2026-08-02T14:40:00Z",
        "eventSource": "bedrock-agentcore.amazonaws.com",
        "eventName": "InvokeAgentRuntime",
        "awsRegion": "us-east-1",
        "sourceIPAddress": "agentcore.amazonaws.com",
        "userAgent": "AgentCoreRuntime/1.0",
        "requestParameters": {
            "agentRuntimeArn": "arn:aws:bedrock-agentcore:us-east-1:111122223333:agent-runtime/TravelAgent-Prod",
            "runtimeSessionId": "session-99x-isolated",
            "runtimeUserId": "user-finance-881",
        },
        "responseElements": {
            "statusCode": 200,
        },
    }

    print(f"  {json.dumps(event, indent=2)}")

def audit_breakdown():
    """Break down the audit trail to show who did what."""
    print_section("Audit Breakdown — Who Did What?")

    print("  1. Who executed the code? (Workload Identity)")
    print("     → arn:aws:sts::111122223333:assumed-role/ProdAgentRole/session-99x-isolated")
    print()
    print("  2. Who initiated the request? (Caller Identity)")
    print("     → user-finance-881 (runtimeUserId)")
    print()
    print("  3. Which agent was invoked? (Resource)")
    print("     → arn:aws:bedrock-agentcore:us-east-1:111122223333:agent-runtime/TravelAgent-Prod")
    print()
    print("  4. When did it happen? (Timestamp)")
    print("     → 2026-08-02T14:40:00Z")
    print()
    print("  Without three-identity separation:")
    print("  - CloudTrail shows 'the role did it' — no caller attribution")
    print("  - Impossible to audit which user triggered which agent action")
    print("  - Security incident response cannot trace blast radius per user")

def show_audit_best_practices():
    """Best practices for agent audit trails."""
    print_section("Audit Best Practices for Agentic Systems")

    practices = [
        "Enable CloudTrail across all regions where AgentCore resources exist",
        "Log runtimeUserId on every InvokeAgentRuntime call for caller attribution",
        "Use distinct workload identities per agent (not one shared role)",
        "Tag all AgentCore resources with owner, environment, and cost center",
        "Set CloudWatch alarms on anomalous invocation patterns (spike in errors, new agents)",
        "Retain CloudTrail logs per compliance requirements (typically 90 days minimum)",
        "Forward AgentCore CloudTrail events to your SIEM for correlation",
    ]

    for i, practice in enumerate(practices, 1):
        print(f"  {i}. {practice}")

def main():
    print("CloudTrail Audit Evidence — Instructor Demo\n")
    show_cloudtrail_event()
    audit_breakdown()
    show_audit_best_practices()
    print_section("Key Takeaways")
    print("  1. Three-identity separation makes every action attributable")
    print("  2. runtimeUserId preserves caller identity in the audit trail")
    print("  3. Without distinct identities, CloudTrail cannot tell who did what")
    print("  4. Forward AgentCore events to your SIEM for enterprise visibility")

if __name__ == "__main__":
    main()
